> 在线讲义：[第 35 章 · 集合通信的实现与代价：环、树、通信库和资源竞争](https://wikieden.github.io/ai-infra-from-constraints/lessons/35-collectives.html) · 本书《从约束推导 AI Infra》

# 第 35 章 · 集合通信的实现与代价：环、树、通信库和资源竞争

配套讲义：仓库里 `ai-infra/notes/lessons/35-collectives.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



环形 / 树形 AllReduce 时间模型、Qwen3-32B 的 TP 单步时间、交点、busbw、实验 6-5 的重叠取自 AII 第 6 章 §6.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
KiB, MiB, GiB = 2**10, 2**20, 2**30
ALPHA, B = 0.822e-6, 450e9  # MSCCL++ 测得的 H100 NVLink 单向延迟、每方向带宽
T_LOCAL = 29.35e-3

**函数 `ring_ar`**：式 (6-9)。

In [ ]:
def ring_ar(n: int, M: float, alpha: float = ALPHA, bw: float = B) -> float:
    """式 (6-9)。"""
    return 2 * (n - 1) * alpha + 2 * (n - 1) * M / (n * bw) if n > 1 else 0.0

**函数 `tree_ar`**：式 (6-10)，n 为 2 的幂。

In [ ]:
def tree_ar(n: int, M: float, alpha: float = ALPHA, bw: float = B) -> float:
    """式 (6-10)，n 为 2 的幂。"""
    return 2 * math.log2(n) * (alpha + M / bw)

**函数 `tp_step`**：式 (6-5)。

In [ ]:
def tp_step(p: int, ar=ring_ar, M: float = 10 * KiB, **kw) -> float:
    """式 (6-5)。"""
    return T_LOCAL / p + 128 * ar(p, M, **kw)

**函数 `ring_tree_crossover`**：两式相等的数据量（字节）。

In [ ]:
def ring_tree_crossover(n: int = 8, alpha: float = ALPHA, bw: float = B) -> float:
    """两式相等的数据量（字节）。"""
    a = 2 * (n - 1) - 2 * math.log2(n)
    b = 2 * math.log2(n) - 2 * (n - 1) / n
    return a * alpha * bw / b

**函数 `busbw`**：小工具。

In [ ]:
def busbw(algbw: float, n: int) -> float:
    return algbw * 2 * (n - 1) / n

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：环形 AllReduce，Qwen3-32B 128K，M = 10 KiB")
for p in (1, 2, 4, 8):
    print(f"  TP{p}：本地 {T_LOCAL / p * 1e3:.2f} ms，128 次归约 {128 * ring_ar(p, 10 * KiB) * 1e3:.2f} ms，单步 {tp_step(p) * 1e3:.2f} ms")
print(f"  TP8 单次 {ring_ar(8, 10 * KiB) * 1e6:.2f} μs；带宽翻倍省 {(tp_step(8) - tp_step(8, bw=2 * B)) * 1e6:.1f} μs，α 减半省 {(tp_step(8) - tp_step(8, alpha=ALPHA / 2)) * 1e3:.2f} ms")

In [ ]:
print("\n第二步：环 vs 树（八卡）")
for M in (10 * KiB, 80 * MiB):
    print(f"  {M / KiB:,.0f} KiB：环 {ring_ar(8, M) * 1e6:,.2f} μs，树 {tree_ar(8, M) * 1e6:,.2f} μs")
print(f"  交点约 {ring_tree_crossover() / KiB:.0f} KiB；α 减半后 {ring_tree_crossover(alpha=ALPHA / 2) / KiB:.0f} KiB")

In [ ]:
print("\n第三步：测量与重叠")
print(f"  16 rank、64 MiB：algbw 143.99 GB/s → busbw {busbw(143.99, 16):.2f} GB/s")
for name, c, k, cc, kk, both in (("4 MiB", 2.67, 11.53, 6.14, 11.52, 11.58), ("64 MiB", 36.92, 11.03, 45.21, 12.89, 45.24)):
    print(f"  {name}：分开 {c + k:.2f} ms，同时 {both:.2f} ms，省 {c + k - both:.2f} ms")

## 3. 代码里才看得到的

1. `ring_tree_crossover` 的解析式：$M^* = \frac{[2(n-1) - 2\log_2 n]\alpha B}{2\log_2 n - 2(n-1)/n}$，八卡是 $8\alpha B/4.25$。
2. 树的时间模型是未分段的二项树；实际 NCCL 的树会分段流水，大数据时比这个快。
3. α 减半那行同时把交点减半：交点和 α 成正比。

## 4. 自己改一改

- 加一个"递归加倍"算法：$\log_2 n$ 轮、每轮 $M$，和环、树比较。
- 把 `tp_step` 的 $M$ 改成 4 个 token（40 KiB），TP8 的单步变多少？
- 画出 $n$ = 4、8、16、32 时的交点。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 收作业四种法：总分给每人、一科给一人、名单拼全、纸条各写；
> 传话围一圈，转两圈人人都有；小数据时全在喊开始；
> 淘汰赛轮少，但每轮传整份；六百八十 KiB 换手；
> 快递车和叉车共一条路，各自单测的时间不能相减。